# 10 — Capstone: the full workflow on HotPotQA

**What you'll learn**

- Nothing new — that is the point. Data → program → metric → baseline → optimize → sealed test, on **real** data, with every skill from notebooks 00–09 and zero new abstractions.
- Along the way: `DataLoader.from_huggingface`, adapting a program's graph to a new data shape, and what changes (and what doesn't) when the world stops being fictional.

*Time: ~40 min. Cost: ~$1–2 depending on which optimizers you run — the priciest notebook of the course. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, browsable any time with `obs.mlflow_ui()`. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()

## The task, and what "real" changes

**HotPotQA** (Yang et al., 2018) is the dataset our Aldervane micro-world was modeled on: multi-hop questions over Wikipedia, bridge entities and all. We use its **distractor** setting: each row ships its own 10 paragraphs — 2 containing the evidence, 8 plausible decoys — so the retrieval problem is *reranking the row's own passages*. Self-contained, no external search service, exactly like our toy setup.

What real data changes: questions are ambiguously worded, gold answers have arbitrary formatting, some examples are simply broken — and scores are lower and noisier. What it does not change: the program shape, the metric discipline, or a single optimizer API. Watching the toy workflow transfer *unedited* is the final lesson of the course.

*(No HuggingFace access? Swap in GSM8K and a one-module `ChainOfThought("question -> answer")` program — the deep import is `from dspy.datasets.gsm8k import GSM8K, gsm8k_metric` — and the rest of this notebook proceeds identically.)*

In [ ]:
from dspy.datasets import DataLoader

dl = DataLoader()
raw = dl.from_huggingface("hotpot_qa", "distractor", split="validation", input_keys=("question",))
print(len(raw), "rows")

row = raw[0]
print(row.question)
print(row.answer)
print(row.context.keys() if hasattr(row.context, "keys") else type(row.context))

> **What you should see:** ~7,400 rows; each has `question`, `answer`, a `context` dict of `{"title": [10 titles], "sentences": [10 sentence-lists]}`, and `supporting_facts` naming the 2 gold titles. (Field names as served by the HF hub — if the schema shifted since authoring, print one row and adjust the flattener below; it is the only schema-dependent code in the notebook.)

Flatten each row into the shape our programs like — `passages: list[str]` plus labels — and carve out modest splits (seeded, so your runs are reproducible):

In [ ]:
import random

def to_example(row):
    passages = [f"{title}: {' '.join(sents)}"
                for title, sents in zip(row.context["title"], row.context["sentences"])]
    return dspy.Example(
        question=row.question,
        answer=row.answer,
        passages=passages,
        gold_titles=list(dict.fromkeys(row.supporting_facts["title"])),
    ).with_inputs("question", "passages")

rng = random.Random(0)
pool = [to_example(r) for r in rng.sample(list(raw), 100)]
train, dev, test = pool[:40], pool[40:70], pool[70:100]
len(train), len(dev), len(test)

Notice the hygiene is muscle memory now: `with_inputs("question", "passages")` — the program sees the question and the 10 paragraphs; `answer` and `gold_titles` stay on the label side, for the metric and (later) for GEPA feedback. Test is sealed until the last cell.

*Sidebar: `dspy.datasets.HotPotQA(...)` also exists as a one-line loader — but it returns question/answer(/gold titles) only, no passages, because it targets the **fullwiki** setting where you bring your own retriever. Our distractor-setting notebook needs the passages, hence `DataLoader` + flattening.*

## The program: notebook 03's graph, one node swapped

Same three predictors, same wiring. The only change: `search()` ranked a global corpus; `rank()` scores the example's own passages (so the IDF is computed per-call over 10 documents — same formula as notebook 03, smaller world):

In [ ]:
# boilerplate: tokenizer (same code as notebook 03)
import json, math, re
from collections import Counter
from pathlib import Path

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

In [ ]:
def rank(passages, query, k=3):
    """Rank an example's own passages against the query — notebook 03's scoring, per call."""
    docs = [set(tokenize(p)) for p in passages]
    idf = Counter()
    for d in docs:
        idf.update(d)
    idf = {t: math.log(len(docs) / df) for t, df in idf.items()}
    q = set(tokenize(query))
    order = sorted(range(len(passages)),
                   key=lambda i: (-sum(idf.get(t, 0.0) for t in q & docs[i]), i))
    return [passages[i] for i in order[:k]]

class HotpotHopQA(dspy.Module):
    """Notebook 03's two-hop graph; retrieval reranks the example's own 10 paragraphs."""

    def __init__(self, k=3):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question, passages):
        q1 = self.hop1(question=question).search_query
        first = rank(passages, q1, k=self.k)
        q2 = self.hop2(question=question, context=first).search_query
        second = rank(passages, q2, k=self.k)
        context = list(dict.fromkeys(first + second))
        pred = self.answer(question=question, context=context)
        return dspy.Prediction(answer=pred.answer, queries=[q1, q2], retrieved=context)

```
              notebook 03 (Aldervane)                 notebook 10 (HotPotQA)
   question -> hop1 -> search(CORPUS) --+     question -> hop1 -> rank(row's passages) --+
            -> hop2 -> search(CORPUS) --+--> answer  -> hop2 -> rank(row's passages) --+--> answer
                        ^^^^^^ the ONE swapped node ^^^^^^
```

## Metric and baseline

The metric transfers verbatim — normalized exact match, same `norm`, same strictness rationale (HotPotQA gold answers are short spans; the official benchmark metric adds token-F1, which is exercise 3):

In [ ]:
def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

In [ ]:
evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8,
                         display_progress=True, display_table=5)

baseline_score = evaluate(HotpotHopQA()).score
baseline_score

> **What you should see:** real data being rude — baselines commonly land around **~20–45 EM** on a 30-question dev slice, with failures messier than Aldervane's (mis-ranked decoys, answers formatted unlike the gold span, genuinely hard hops). Skim the failure table with your notebook-04 eyes before optimizing: which family dominates, retrieval or answering?
>
> Capstone leaderboard:
>
> | program | dev EM (30 q) | notebook |
> |---|---|---|
> | `HotpotHopQA()` zero-shot | *yours* | 10 |

## Optimize — in cost order

First rung of the ladder, exactly as on the toy task: bootstrap demos with the trace-gated metric pattern from notebook 05.

In [ ]:
def qa_metric(example, pred, trace=None):
    """Exact match; strict gate when bootstrapping (the notebook 05 pattern)."""
    correct = norm(pred.answer) == norm(example.answer)
    if trace is not None:
        return correct
    return float(correct)

boot = dspy.BootstrapFewShot(
    metric=qa_metric, max_bootstrapped_demos=4, max_labeled_demos=4,
).compile(HotpotHopQA(), trainset=train)

boot_score = evaluate(boot).score
print(f"baseline {baseline_score} -> bootstrap {boot_score}")

> **What you should see:** usually a solid single-digit-to-teens jump, from the same two mechanisms as before (bridge-entity hop-2 demos + terse-answer style). Inspect `boot.hop2.demos` and check — Wikipedia bridge entities this time, but the same learned behavior.

**Choose your fighter for the second rung.** Both cells below work; run one (or both, budget permitting — each is several hundred cache-cold LM calls, a few minutes):

- **MIPROv2 light** — the default serious choice (notebook 06).
- **GEPA light** — if you want the feedback loop; note how the toy feedback metric transfers: `gold_titles` vs. retrieved titles instead of `gold_ids` vs. `retrieved_ids`.

In [ ]:
mipro_prog = dspy.MIPROv2(
    metric=qa_metric, auto="light", max_bootstrapped_demos=4, max_labeled_demos=4,
).compile(HotpotHopQA(), trainset=train, valset=dev)

mipro_score = evaluate(mipro_prog).score
print(f"baseline {baseline_score} -> bootstrap {boot_score} -> MIPROv2 {mipro_score}")

In [ ]:
# OPTIONAL (extra cost): the GEPA rung — notebook 07's feedback metric, retargeted in ~5 lines.
def hotpot_feedback_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    score = float(norm(pred.answer) == norm(gold.answer))
    retrieved_titles = {p.split(":", 1)[0] for p in pred.retrieved}
    missing = [t for t in gold.gold_titles if t not in retrieved_titles]
    if missing:
        feedback = (f"Retrieval failure: gold paragraph(s) titled {missing} were never ranked "
                    f"into the context. Queries used: {pred.queries}. The second query should "
                    "name the bridge entity found in hop 1, not restate the question.")
    elif not score:
        feedback = (f"Evidence was retrieved but {pred.answer!r} != {gold.answer!r}. Answer "
                    "with the shortest exact span from the passages, no extra words.")
    else:
        feedback = "Correct."
    return dspy.Prediction(score=score, feedback=feedback)

gepa_prog = dspy.GEPA(
    metric=hotpot_feedback_metric, auto="light",
    reflection_lm=dspy.LM(STRONG_MODEL, temperature=1.0, max_tokens=8000),
    track_stats=True,
).compile(HotpotHopQA(), trainset=train, valset=dev)

gepa_score = evaluate(gepa_prog).score
print(f"GEPA {gepa_score}")

> **What you should see:** the familiar arcs — logged trials, then dev scores that usually clear the bootstrap rung; read the evolved instructions (`named_predictors()`) and you'll find Wikipedia-flavored versions of the same discovered rules (bridge-entity queries, span-exact answers). Update the leaderboard with whatever you ran.

## The final ritual

In [ ]:
best_program = mipro_prog        # <- whichever leads YOUR leaderboard

test_evaluate = dspy.Evaluate(devset=test, metric=em_metric, num_threads=8, display_progress=True)
print("sealed test EM:", test_evaluate(best_program).score)

Path("artifacts").mkdir(exist_ok=True)
best_program.save("artifacts/10_final.json")
print("saved artifacts/10_final.json — go read your learned parameters one last time")

> **What you should see:** one honest number (expect it near, often a bit under, your dev score — you tuned on dev), and one JSON file. Open the file. Every trick this course taught is sitting in it as plain text: per-predictor demos with learned intermediate steps, evolved instructions encoding discovered rubrics. That file — plus 40 lines of program — is the entire deliverable of an optimized LLM system in DSPy.

## Recap: the course, in one table

| You learned | Notebook | The API that carries it |
|---|---|---|
| LM setup, caching, the PyTorch analogy | 00 | `dspy.LM`, `dspy.configure`, `dspy.context` |
| Typed I/O contracts | 01 | `dspy.Signature`, `dspy.Predict`, `inspect_history` |
| Prompts are rendered; θ = instructions + demos | 02 | adapters, `dspy.Example`, `LabeledFewShot`, `save()` |
| Programs = LM nodes + Python, define-by-run | 03 | `dspy.Module`, `forward`, `named_predictors()` |
| The loss function and the eval loop | 04 | metrics, `trace`, `dspy.Evaluate`, `EvaluationResult` |
| Self-generated demos | 05 | `BootstrapFewShot(+RS)`, the `trace` gate |
| Machine-written instructions | 06 | `COPRO`, `MIPROv2(auto=...)` |
| Textual feedback as gradient | 07 | `GEPA`, 5-arg metrics, `Prediction(score, feedback)` |
| Dynamic graphs & runtime compute | 08 | `ReAct`, tools, `Refine`/`BestOfN` |
| The full zoo and when to use what | 09 | `BootstrapFinetune`, `BetterTogether`, `Ensemble`, chooser |
| All of it, on real data | 10 | nothing new — that was the point |
| Seeing inside a running program | 11 | `BaseCallback`, `mlflow.dspy.autolog`, Phoenix |

**Where to go next:** [dspy.ai](https://dspy.ai) for docs and tutorials; the papers cited throughout (each optimizer notebook links its arXiv source); notebook 11 for the observability layer (tracing, and optimizer runs you can browse); `dspy.streamify`/`asyncify` when you take a program to production; and your own task — the workflow you just ran twice is the workflow.

## Exercises

1. Re-run the baseline with a stronger task model (swap `MODEL`, rerun from the config cell). How much of the optimizers' edge evaporates when the base model is smarter — and what does that tell you about when optimization pays?
2. Extend `hotpot_feedback_metric` to detect the *decoy trap*: when a retrieved-but-wrong paragraph shares its title's first word with a gold title, say so in the feedback. Does GEPA's evolved `hop2` instruction start warning about near-duplicate titles?
3. Score your best program with `dspy.evaluate.SemanticF1()` alongside exact match. Which examples does EM call wrong that SemanticF1 forgives — and after reading ten of them, which metric do you *believe*?

**Next up:** one last module — *watching* all of this run. Notebook 11 builds a tracer from DSPy's callback hooks, then lets MLflow record your programs and optimizer compiles (with Arize Phoenix as the nicer-UI option for trace inspection).